# Phase 1.3 — Data Cleaning & Preparation

This notebook implements the data cleaning rules defined in:
`01_data_preparation/docs/cleaning_rules.md`

Goals:
- Preserve raw data
- Parse timestamps
- Create numeric columns
- Flag (not fix) data quality issues
- Generate an analytics-ready dataset

No aggregation, imputation, or modeling is performed in this phase.

In [2]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [3]:
RAW_DATA_PATH = "../../00_foundation_crypto_ingestion/data_samples/Crypto_Data.xlsx"

In [4]:
df_raw = pd.read_excel(RAW_DATA_PATH)

In [5]:
print("Raw dataset shape:", df_raw.shape)

Raw dataset shape: (52827, 14)


In [6]:
df_raw.head()

,Coin,Symbol,Trend Emoji,Trend,24h Change,Current Price,Market Cap,24h Volume,Global Rank,ATH Insight,24h Range,Volatility,Chart Link,Updated At
0,Bonk,BONK,🚀,Bullish 🔥,0.2646,₹0.003,"₹2,60,66,58,87,002","₹2,52,01,53,61,536 🔺",45,🔴 Far Below ATH,🔼 Near 24h High (100.0%),🔥 High Volatility,https://www.coingecko.com/en/coins/bonk,2025-07-17 11:30:57
1,XRP,XRP,🚀,Bullish 🔥,0.0775,₹269.68,"₹1,59,46,17,15,79,695","₹7,75,88,21,92,814 🔻",3,🟢 Near ATH,🔼 Near 24h High (100.0%),🟦 Medium Volatility,https://www.coingecko.com/en/coins/ripple,2025-07-17 11:30:57
2,Lido Staked Ether,STETH,🚀,Bullish 🔥,0.0734,"₹2,89,186","₹26,34,46,84,20,224","₹9,85,41,54,863 🔻",9,🟡 Watch Zone,🔼 Near 24h High (84.3%),🟦 Medium Volatility,https://www.coingecko.com/en/coins/staked-ether,2025-07-17 11:30:57
3,Ethereum,ETH,🚀,Bullish 🔥,0.0733,"₹2,89,555","₹3,49,47,15,10,44,618","₹49,43,93,85,20,583 🔺",2,🟡 Watch Zone,🔼 Near 24h High (83.6%),🟦 Medium Volatility,https://www.coingecko.com/en/coins/ethereum,2025-07-17 11:30:57
4,Wrapped eETH,WEETH,🚀,Bullish 🔥,0.0730,"₹3,10,016","₹7,85,14,96,88,813","₹1,41,94,83,664 🔻",21,🟡 Watch Zone,🔼 Near 24h High (83.3%),🟦 Medium Volatility,https://www.coingecko.com/en/coins/wrapped-eeth,2025-07-17 11:30:57


In [7]:
df = df_raw.copy()

## Timestamp Parsing

The `Updated At` column is stored as text and must be converted into a
proper datetime column to enable time-based analysis.

The original column will be preserved for auditability.

In [8]:
df["updated_at_parsed"] = pd.to_datetime(
    df["Updated At"],
    errors="coerce"
)

In [9]:
df[["Updated At", "updated_at_parsed"]].head()

,Updated At,updated_at_parsed
0,2025-07-17 11:30:57,2025-07-17 11:30:57
1,2025-07-17 11:30:57,2025-07-17 11:30:57
2,2025-07-17 11:30:57,2025-07-17 11:30:57
3,2025-07-17 11:30:57,2025-07-17 11:30:57
4,2025-07-17 11:30:57,2025-07-17 11:30:57


In [10]:
df["updated_at_parsed"].isna().sum()

np.int64(0)

In [11]:
df["Updated At"].dtype, df["updated_at_parsed"].dtype

(dtype('<M8[ns]'), dtype('<M8[ns]'))

In [12]:
# Rename original column to make intent explicit
df = df.rename(columns={"Updated At": "updated_at_raw"})

# Use the validated parsed column as the analytical timestamp
df = df.rename(columns={"updated_at_parsed": "updated_at"})

In [13]:
duplicate_mask = df.duplicated(
    subset=["Coin", "updated_at"],
    keep="first"
)

duplicate_mask.sum()

np.int64(0)

In [14]:
def extract_numeric(value):
    if pd.isna(value):
        return np.nan
    
    cleaned = re.sub(r"[^0-9.\-]", "", str(value))
    
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

In [15]:
df["current_price_numeric"] = df["Current Price"].apply(extract_numeric)

In [16]:
df["market_cap_numeric"] = df["Market Cap"].apply(extract_numeric)
df["volume_24h_numeric"] = df["24h Volume"].apply(extract_numeric)
df["price_change_24h_numeric"] = df["24h Change"].apply(extract_numeric)
df["global_rank_numeric"] = df["Global Rank"].apply(extract_numeric)

In [17]:
df[
    [
        "Current Price",
        "current_price_numeric",
        "Market Cap",
        "market_cap_numeric",
        "24h Volume",
        "volume_24h_numeric"
    ]
].head(10)

,Current Price,current_price_numeric,Market Cap,market_cap_numeric,24h Volume,volume_24h_numeric
0,₹0.003,0.003,"₹2,60,66,58,87,002",2.606659e+11,"₹2,52,01,53,61,536 🔺",2.520154e+11
1,₹269.68,269.680,"₹1,59,46,17,15,79,695",1.594617e+13,"₹7,75,88,21,92,814 🔻",7.758822e+11
2,"₹2,89,186",289186.000,"₹26,34,46,84,20,224",2.634468e+12,"₹9,85,41,54,863 🔻",9.854155e+09
3,"₹2,89,555",289555.000,"₹3,49,47,15,10,44,618",3.494715e+13,"₹49,43,93,85,20,583 🔺",4.943939e+12
4,"₹3,10,016",310016.000,"₹7,85,14,96,88,813",7.851497e+11,"₹1,41,94,83,664 🔻",1.419484e+09
5,₹0.001,0.001,"₹4,85,54,05,26,253",4.855405e+11,"₹6,52,96,41,80,631 🔺",6.529642e+11
6,"₹1,434.51",1434.510,"₹9,72,28,32,30,864",9.722832e+11,"₹78,46,56,76,185 🔺",7.846568e+10
7,₹757.23,757.230,"₹4,54,47,56,62,881",4.544757e+11,"₹87,36,90,23,917 🔺",8.736902e+10
8,₹20.71,20.710,"₹8,77,22,53,23,946",8.772253e+11,"₹58,94,39,77,530 🔺",5.894398e+10
9,₹65.95,65.950,"₹23,82,39,26,45,746",2.382393e+12,"₹1,28,64,94,19,430 🔺",1.286494e+11


In [18]:
df = df.sort_values(["Coin", "updated_at"])

In [19]:
df["time_diff_minutes"] = (
    df.groupby("Coin")["updated_at"]
      .diff()
      .dt.total_seconds() / 60
)

In [20]:
df["interval_gap_flag"] = df["time_diff_minutes"] > 15

In [21]:
df["interval_gap_flag"].value_counts(dropna=False)

interval_gap_flag
True     26652
False    26175
Name: count, dtype: int64

In [22]:
def categorize_gap(minutes):
    if pd.isna(minutes):
        return "first_record"
    elif minutes <= 15:
        return "<=15_min"
    elif minutes <= 30:
        return "15_30_min"
    elif minutes <= 45:
        return "31_45_min"
    elif minutes <= 60:
        return "46_60_min"
    else:
        return ">60_min"

In [23]:
df["gap_category"] = df["time_diff_minutes"].apply(categorize_gap)

In [24]:
df["gap_category"].value_counts()

gap_category
<=15_min        26110
15_30_min       15867
>60_min          6050
31_45_min        3554
46_60_min        1181
first_record       65
Name: count, dtype: int64

In [25]:
df["pipeline_type"] = np.select(
    [
        (df["updated_at"] >= "2025-07-17 11:30:57") & (df["updated_at"] <= "2025-08-05 12:00:33"),
        (df["updated_at"] >= "2025-08-05 12:30:41") & (df["updated_at"] <= "2025-08-31 00:00:10"),
        (df["updated_at"] >= "2025-08-31 00:13:19") & (df["updated_at"] <= "2025-10-04 14:45:03")
    ],
    [
        "n8n_local",
        "n8n_docker",
        "github_actions"
    ],
    default="unknown"
)

In [26]:
df["pipeline_type"].value_counts()

pipeline_type
github_actions    28326
n8n_docker        17970
n8n_local          6531
Name: count, dtype: int64

In [27]:
gap_pipeline_ct = (
    df
    .groupby(["pipeline_type", "gap_category"])
    .size()
    .reset_index(name="row_count")
)

gap_pipeline_ct.sort_values(["pipeline_type", "row_count"], ascending=[True, False])

,pipeline_type,gap_category,row_count
0,github_actions,15_30_min,11559
3,github_actions,<=15_min,10134
4,github_actions,>60_min,3280
1,github_actions,31_45_min,2647
2,github_actions,46_60_min,689
5,github_actions,first_record,17
9,n8n_docker,<=15_min,13432
10,n8n_docker,>60_min,1885
6,n8n_docker,15_30_min,1862
7,n8n_docker,31_45_min,483


In [28]:
gap_pipeline_pivot = (
    df
    .pivot_table(
        index="pipeline_type",
        columns="gap_category",
        values="updated_at",
        aggfunc="count",
        fill_value=0
    )
)

gap_pipeline_pivot

gap_category,15_30_min,31_45_min,46_60_min,<=15_min,>60_min,first_record
pipeline_type,,,,,,
github_actions,11559,2647,689,10134,3280,17
n8n_docker,1862,483,305,13432,1885,3
n8n_local,2446,424,187,2544,885,45


In [29]:
df.dtypes

Coin                                object
Symbol                              object
Trend Emoji                         object
Trend                               object
24h Change                         float64
Current Price                       object
Market Cap                          object
24h Volume                          object
Global Rank                          int64
ATH Insight                         object
24h Range                           object
Volatility                          object
Chart Link                          object
updated_at_raw              datetime64[ns]
updated_at                  datetime64[ns]
current_price_numeric              float64
market_cap_numeric                 float64
volume_24h_numeric                 float64
price_change_24h_numeric           float64
global_rank_numeric                float64
time_diff_minutes                  float64
interval_gap_flag                     bool
gap_category                        object
pipeline_ty

In [30]:
df["gap_category"] = df["gap_category"].astype("category")
df["pipeline_type"] = df["pipeline_type"].astype("category")

In [31]:
OUTPUT_PATH = "../outputs/crypto_cleaned_v1.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("Final validated dataset saved.")

Final validated dataset saved.
